In [1]:
# AI Use notice: Gemini was used for prints and debugging

In [2]:
import pandas as pd

df = pd.read_csv("../../data/cs2_tier1_games.csv", encoding='latin1')
print(f"Initial shape: {df.shape}")

Initial shape: (9072, 99)


In [3]:
df = df[df['is_total'] == False].copy()
print(f"Shape after filtering: {df.shape}")

Shape after filtering: (6132, 99)


In [4]:
# drop stats not available to predict the game before it happens
stats_suffixes = ['_kills', '_deaths', '_assists', '_adr', '_kast', '_kddiff']
cols_to_drop = [col for col in df.columns if any(col.endswith(s) for s in stats_suffixes)]

# drop scores and other post-game info.
cols_to_drop += [
    'Unnamed: 0', 'match_id', 'game_id', 'is_total', 
    'score1_match', 'score2_match', 'score1_game', 'score2_game',
    'games_played', 'map_id'
]

df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns. Remaining: {df.shape[1]}")

Dropped 70 columns. Remaining: 29


In [5]:
# convert dates for space efficiency
df['datetime'] = pd.to_datetime(df['datetime'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day_of_week'] = df['datetime'].dt.dayofweek
df['hour'] = df['datetime'].dt.hour
df = df.drop(columns=['datetime'])

In [6]:
# Drop player name strings as they are not necessary
name_cols = [col for col in df.columns if 'player' in col and '_id' not in col]
df = df.drop(columns=name_cols)

from sklearn.preprocessing import LabelEncoder

categorical = ['tournament', 'team1', 'team2', 'map_name']
le = LabelEncoder()

for col in categorical:
    df[col] = le.fit_transform(df[col].astype(str))

df = df.dropna()

print(f"Final shape: {df.shape}")
df.head()

Final shape: (5569, 22)


,tournament,team1_id,team1,team2_id,team2,bestOf,map_name,team1_win,team1_player1_id,team1_player2_id,...,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id,year,month,day_of_week,hour
0,10,288898,122,288899,199,5.0,3,0,1401.0,169.0,...,7201.0,19.0,550.0,1549.0,1443.0,146.0,2026,3,6,13
1,10,288898,122,288899,199,5.0,1,0,1401.0,169.0,...,7201.0,19.0,550.0,1549.0,1443.0,146.0,2026,3,6,13
3,10,288894,137,288895,128,3.0,3,1,2988.0,266.0,...,16113.0,1401.0,169.0,1834.0,2934.0,7201.0,2026,3,5,19
4,10,288894,137,288895,128,3.0,4,0,2988.0,266.0,...,16113.0,1401.0,169.0,1834.0,2934.0,7201.0,2026,3,5,19
5,10,288894,137,288895,128,3.0,2,0,2988.0,266.0,...,16113.0,1401.0,169.0,1834.0,2934.0,7201.0,2026,3,5,19


In [7]:
df.to_csv("../../data/cleaned_cs2_games_pregame.csv", index=False)
print("Saved to data/cleaned_cs2_games_pregame.csv")

Saved to data/cleaned_cs2_games_pregame.csv
